In [ ]:
import logging
from dataclasses import asdict
import sys
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from datasets import load_dataset
from IPython.display import display

logging.basicConfig(level=logging.INFO)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "SWE-Gym/OpenHands-Sampled-Trajectories",
    split="train.raw",
)

df = dataset.to_pandas().merge(
    load_dataset("SWE-Gym/SWE-Gym", split="train").to_pandas(),
    on="instance_id",
    how="left",
)

print(f"Trajectories: {len(dataset):,} | Combined rows: {len(df):,}")
print(df.columns.tolist())

In [ ]:
sample_index = 0
sample = df.iloc[sample_index]
print(f"[STAGE] Inspect record {sample_index}")
print("✓ Fields:", df.columns)
display(pd.Series({key: sample.get(key) for key in ["instance_id", "run_id", "resolved"]}, name="value").to_frame())
messages = sample.get("messages")
tools = sample.get("tools")
print("Messages:", len(messages) if messages is not None else 0)
print("Available tools:", [tool.get("function", {}).get("name") for tool in (tools if tools is not None else [])])

#### 4.b Inspect messages in order

Each row is a recorded message. `tool_call_id` links a tool response to its call when present.

In [72]:
sample['resolved'], sample['test_result']

(np.False_,
 {'apply_patch_output': None,
  'git_patch': '',
  'report': {'empty_generation': True,
   'error_eval': False,
   'failed_apply_patch': False,
   'resolved': False,
   'test_timeout': False},
  'test_output': None})

In [73]:
import json

message_df = pd.DataFrame(list(messages) if messages is not None else [])
message_df["message_index"] = message_df.index
message_df = message_df.explode("tool_calls", ignore_index=False)

call_details = pd.json_normalize([
    call if isinstance(call, dict) else {}
    for call in message_df.pop("tool_calls")
]).rename(columns={
    "id": "call_id",
    "type": "call_type",
    "function.name": "call_name",
    "function.arguments": "call_arguments",
})

message_df = pd.concat(
    [message_df.reset_index(drop=True), call_details],
    axis=1,
)

argument_dicts = [
    json.loads(value) if isinstance(value, str) else value
    for value in message_df.pop("call_arguments")
]
argument_df = pd.json_normalize([
    value if isinstance(value, dict) else {}
    for value in argument_dicts
]).add_prefix("arg_")

tools_df = pd.json_normalize(sample["tools"]).rename(
    columns={"function.name": "call_name"}
)

message_df = pd.concat([message_df, argument_df], axis=1)
message_df = message_df.merge(tools_df, on="call_name", how="left")

display(message_df), display(tools_df)

,content,function_call,name,role,tool_call_id,message_index,call_id,index,call_type,call_name,...,function.parameters.properties.insert_line.type,function.parameters.properties.new_str.description,function.parameters.properties.new_str.type,function.parameters.properties.old_str.description,function.parameters.properties.old_str.type,function.parameters.properties.path.description,function.parameters.properties.path.type,function.parameters.properties.view_range.description,function.parameters.properties.view_range.items.type,function.parameters.properties.view_range.type
0,You are a helpful assistant that can interact ...,None,None,system,None,0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,<uploaded_files>\n/workspace/getmoto__moto__3....,None,None,user,None,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,None,None,assistant,None,2,call_yvYHZVbXjUPsvihtKQVVOtI6,NaN,function,str_replace_editor,...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter of `view` command when `pat...,integer,array
3,OBSERVATION:\nHere's the files and directories...,None,str_replace_editor,tool,call_yvYHZVbXjUPsvihtKQVVOtI6,3,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,,None,None,assistant,None,4,call_1pJhJa9YD2hFdbWDIUhuqEIV,NaN,function,str_replace_editor,...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter of `view` command when `pat...,integer,array
5,OBSERVATION:\nHere's the files and directories...,None,str_replace_editor,tool,call_1pJhJa9YD2hFdbWDIUhuqEIV,5,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,None,None,assistant,None,6,call_rloEZJfziaKCxcYAW5Tud0mC,NaN,function,str_replace_editor,...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter of `view` command when `pat...,integer,array
7,OBSERVATION:\nHere's the files and directories...,None,str_replace_editor,tool,call_rloEZJfziaKCxcYAW5Tud0mC,7,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,It seems there was an issue with viewing the c...,None,None,assistant,None,8,call_mP50aGtV3KdCx9ggln57iIvh,NaN,function,str_replace_editor,...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter of `view` command when `pat...,integer,array
9,OBSERVATION:\nHere's the files and directories...,None,str_replace_editor,tool,call_mP50aGtV3KdCx9ggln57iIvh,9,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,type,function.description,call_name,function.parameters.properties.command.description,function.parameters.properties.command.enum,function.parameters.properties.command.type,function.parameters.properties.file_text,function.parameters.properties.insert_line,function.parameters.properties.new_str,function.parameters.properties.old_str,...,function.parameters.properties.insert_line.type,function.parameters.properties.new_str.description,function.parameters.properties.new_str.type,function.parameters.properties.old_str.description,function.parameters.properties.old_str.type,function.parameters.properties.path.description,function.parameters.properties.path.type,function.parameters.properties.view_range.description,function.parameters.properties.view_range.items.type,function.parameters.properties.view_range.type
0,function,Execute a bash command in the terminal.\n* Lon...,execute_bash,The bash command to execute. Can be empty to v...,None,string,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,function,Finish the interaction when the task is comple...,finish,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,function,"Custom editing tool for viewing, creating and ...",str_replace_editor,The commands to run. Allowed options are: `vie...,"[view, create, str_replace, insert, undo_edit]",string,NaN,NaN,NaN,NaN,...,integer,Optional parameter of `str_replace` command co...,string,Required parameter of `str_replace` command co...,string,"Absolute path to file or directory, e.g. `/rep...",string,Optional parameter of `view` command when `pat...,integer,array


(None, None)

In [65]:
def extract_tool_interactions(messages):
    """Extract tool calls and match them to their responses."""
    responses = {}

    # Index tool responses by the ID of the call they answer.
    for message_index, message in enumerate(messages):
        if not isinstance(message, dict):
            continue

        call_id = message.get("tool_call_id")
        if message.get("role") == "tool" and call_id:
            responses.setdefault(call_id, (message_index, message.get("content")))

    interactions = []

    for message_index, message in enumerate(messages):
        if not isinstance(message, dict):
            continue

        for call in message.get("tool_calls") or []:
            if not isinstance(call, dict):
                continue

            call_id = call.get("id")
            function = call.get("function") or {}
            response = responses.get(call_id)

            interactions.append({
                "call_id": call_id,
                "tool_name": function.get("name"),
                "arguments": function.get("arguments"),
                "call_message_index": message_index,
                "response_message_index": response[0] if response else None,
                "response_content": response[1] if response else None,
                "matched": response is not None,
            })

    return interactions

In [66]:
interactions = extract_tool_interactions(messages)
display(pd.DataFrame(interactions))

,call_id,tool_name,arguments,call_message_index,response_message_index,response_content,matched
0,call_yvYHZVbXjUPsvihtKQVVOtI6,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",2,3.0,OBSERVATION:\nHere's the files and directories...,True
1,call_1pJhJa9YD2hFdbWDIUhuqEIV,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",4,5.0,OBSERVATION:\nHere's the files and directories...,True
2,call_rloEZJfziaKCxcYAW5Tud0mC,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",6,7.0,OBSERVATION:\nHere's the files and directories...,True
3,call_mP50aGtV3KdCx9ggln57iIvh,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",8,9.0,OBSERVATION:\nHere's the files and directories...,True
4,call_2RbnOQNCs5tCZIe3NLJwzDxT,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",10,11.0,OBSERVATION:\nHere's the files and directories...,True
5,call_ex9taEHTSftS9e36qCdpxe3m,str_replace_editor,"{""command"":""view"",""path"":""/workspace/getmoto__...",12,NaN,None,False


In [69]:
print(messages[12])
print("Total messages:", len(messages))

{'content': None, 'function_call': None, 'name': None, 'role': 'assistant', 'tool_call_id': None, 'tool_calls': array([{'function': {'arguments': '{"command":"view","path":"/workspace/getmoto__moto__3.1"}', 'name': 'str_replace_editor'}, 'id': 'call_ex9taEHTSftS9e36qCdpxe3m', 'index': None, 'type': 'function'}],
      dtype=object)}
Total messages: 13


In [ ]:
sample['test_result']

{'apply_patch_output': None,
 'git_patch': '',
 'report': {'empty_generation': True,
  'error_eval': False,
  'failed_apply_patch': False,
  'resolved': False,
  'test_timeout': False},
 'test_output': None}